<a href="https://colab.research.google.com/github/Vikas2826q/Python-Internship/blob/main/CMOS%20parametric%20sweep%20using%20Python" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!apt-get -qq update && apt-get -qq install -y ngspice

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package ngspice.
(Reading database ... 122807 files and directories currently installed.)
Preparing to unpack .../ngspice_42+ds-3build1_amd64.deb ...
Unpacking ngspice (42+ds-3build1) ...
Setting up ngspice (42+ds-3build1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [2]:
import subprocess

template = """* CMOS inverter delay test
.model nm nmos (level=1 vto=0.5 kp=200u)
.model pm pmos (level=1 vto=-0.5 kp=100u)
Vdd vdd 0 1.8
Vin in 0 PULSE(0 1.8 1n 100p 100p 5n 10n)
M1 out in vdd vdd pm w=2u l=0.18u
M2 out in 0 0 nm w=1u l=0.18u
Cl out 0 CLOAD
.tran 10p 20n
.measure tran tpd TRIG v(in) VAL=0.9 RISE=1 TARG v(out) VAL=0.9 FALL=1
.end
"""

for c in ["10f", "50f", "100f", "200f"]:
    open("run.cir", "w").write(template.replace("CLOAD", c))     # fill in the load and save
    result = subprocess.run(["ngspice", "-b", "run.cir"], capture_output=True, text=True)
    for line in result.stdout.splitlines():
        if "tpd" in line:
            print(c, "->", line)

10f -> tpd                 =  3.153828e-11 targ=  1.081538e-09 trig=  1.050000e-09
50f -> tpd                 =  7.461054e-11 targ=  1.124611e-09 trig=  1.050000e-09
100f -> tpd                 =  1.231669e-10 targ=  1.173167e-09 trig=  1.050000e-09
200f -> tpd                 =  2.203762e-10 targ=  1.270376e-09 trig=  1.050000e-09
